In [2]:
from turtledemo.round_dance import stop

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

# Intro to tensors

In [3]:
# Working with tensors - jednodimenzionální je skalár, dvou je matice, více je vícedimenzionálně indexovaná data, třeba u barevných obrázků x\in\R^(3xHxW) (3 protože RGB)
# pytorch vidí vše jako batch
a = torch.rand(10, 5) # náhodná matice
b = torch.rand(5, 17)
mult = torch.matmul(a, b) # maticový násobení
print(mult.shape)

torch.Size([10, 17])


In [4]:
# Working with tensors (batch multiplication)
a = torch.rand(16, 10, 5) # první vidí jako batch, další jako maticový
b = torch.rand(16, 5, 17)
mult = torch.matmul(a, b) # maticový násobení pro batch - neboli 16 matic 10x17
print(mult.shape)

torch.Size([16, 10, 17])


In [5]:
# Gradient example
a = torch.Tensor([1, 2, 3])
b = torch.Tensor([4, 5, 6])
c = torch.Tensor([7, 8, 9])

a.requires_grad = True # bude si pamatovat gradienty (důležitý pro backward pass)
b.requires_grad = True
c.requires_grad = True

torch.sum((a * b) + c).backward()
print(a.grad), print(b.grad), print(c.grad)

tensor([4., 5., 6.])
tensor([1., 2., 3.])
tensor([1., 1., 1.])


(None, None, None)

In [6]:
a = torch.Tensor(torch.rand(1, 4))
a.requires_grad = True
b = a**2 # umocním na druhou (po prvcích)
c = b*2 #
d = c.mean()
e = c.sum()

In [7]:
d.backward(retain_graph=True) # fine, spočítá gradienty všech tensorů který maj true a byly použity pro výpočet d, neboli d vzhledem k c, b, a a uložili se do a.grad, b.grad, c. grad
e.backward(retain_graph=True) # fine
d.backward() # also fine, vlastně to stejný, ale zapomene se ten výpočetní graf, zase se přičtou ty gradienty
e.backward() # error will occur!


# backward přičítá!!!! gradienty k proměnným, které jsou ve výpočetním grafu

RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

# Train model

In [8]:
# Create random dataset. Every Dataset has to implement __len__ and __getitem__
class SyntheticDataset(Dataset):
    def __init__(self, num_samples=1000, input_dim=20):
        self.X = torch.rand(num_samples, input_dim)
        self.y = (torch.mean(self.X, dim=1) > 1/2).type(torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [9]:
# Create model, pomocí subclass metody torch.nn.module
class MultiLayerNet(nn.Module): # bude dědit od nn.module
    def __init__(self, input_dim=20, hidden_dims=[64, 32], output_dim=2): # do konsstruktoru si ukládaám operace který někdy během forward pass použiju
        super(MultiLayerNet, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dims[0]) # jedna to maticový násobení
        self.bn1 = nn.BatchNorm1d(hidden_dims[0])
        self.activation1 = nn.ReLU() #aktivace
            
        self.fc2 = nn.Linear(hidden_dims[0], hidden_dims[1]) #zase lineární
        self.bn2 = nn.BatchNorm1d(hidden_dims[1])
        self.activation2 = nn.ReLU() #aktivace
            
        self.head = nn.Linear(hidden_dims[1], output_dim)

    def forward(self, x): # forward pass, automaticky mám i backward
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.activation1(x)
        
        x = self.fc2(x)
        x = self.bn2(x)
        x = self.activation2(x)
        x = self.head(x)
        return x

In [10]:
# Training loop
def train(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for batch_idx, (inputs, targets) in enumerate(dataloader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward() # u všech parametrů modelu se přičtou gradienty
        optimizer.step() # update pro theta_(n+1) a šáhne si pro ty spočítaný gradienty


        total_loss += loss.item()
    avg_loss = total_loss / len(dataloader)
    print(f"Train Loss: {avg_loss:.4f}")

In [11]:
# Validation loop, validace modelu
def validate(model, dataloader, criterion, device):
    model.eval() # ted budeš evaluovat, nebudeš trénovat
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad(): # ted co se uděje, tak nebudeš počítat gradienty, šetří se paměť
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            total_loss += loss.item()

            preds = outputs.argmax(dim=1)
            correct += (preds == targets).sum().item()
            total += targets.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100.0 * correct / total
    print(f"Validation Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")
    return avg_loss, accuracy

In [12]:
# Hyperparameters and setup
input_dim = 20
batch_size = 32
epochs = 5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset and split
dataset = SyntheticDataset(num_samples=1000, input_dim=input_dim)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [13]:
# Model, criterion, optimizer
model = MultiLayerNet(input_dim=input_dim).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

In [18]:
# Full training + validation loop, dopsán TASK
x = float("inf")
for epoch in range(epochs):
    print(f"\nEpoch {epoch + 1}/{epochs}")
    train(model, train_loader, criterion, optimizer, device)
    y = validate(model, val_loader, criterion, device)
    val_loss = y[0]
    if (epoch + 1)>1 and val_loss >= x:
        print(val_loss)
        break
    x = val_loss


Epoch 1/5
Train Loss: 0.0758
Validation Loss: 0.1361, Accuracy: 93.50%

Epoch 2/5
Train Loss: 0.0734
Validation Loss: 0.1444, Accuracy: 93.00%
0.1443976584289755


# Tasks
- add early stopping
- play with number of parameters in each layer, activation function and regularization parameters and observe how the training changes